PySpark coding interview question: 
### find students with the same marks in math and chemistry  

In this video, we tackle a PySpark coding interview question commonly asked in Data Engineering interviews. We will:  

✅ Generate a dataset of students with marks in multiple subjects.  
✅ Implement an optimized PySpark solution to find students having the same marks in Math and Chemistry.  
✅ Use window functions (LAG) and filtering for better performance, avoiding expensive operations like `pivot()`.  
✅ Discuss optimization techniques to improve PySpark query performance.  

This question is often asked in interviews at companies like PWC, KPIT, Accenture, and more!  


In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
    .builder
    .appName("spark-csv-data")
    .master("local[*]").config("spark.ui.port", "4040")
    .getOrCreate()
)

#
print(spark.sparkContext.uiWebUrl)

#
spark

In [0]:
data = [
    (101, "Alice", "Math", 85),
    (101, "Alice", "Chemistry", 85),
    (101, "Alice", "Physics", 78),
    (102, "Bob", "Math", 90),
    (102, "Bob", "Physics", 88),
    (103, "Charlie", "Math", 75),
    (103, "Charlie", "Chemistry", 75),
    (104, "David", "Math", 88),
    (104, "David", "Chemistry", 88),
    (104, "David", "Physics", 95),
    (105, "Eve", "Chemistry", 91),
]

columns = ["Student_ID", "Name", "Subject", "Marks"]
df = spark.createDataFrame(data, columns)
df.display() 


Student_ID,Name,Subject,Marks
101,Alice,Math,85
101,Alice,Chemistry,85
101,Alice,Physics,78
102,Bob,Math,90
102,Bob,Physics,88
103,Charlie,Math,75
103,Charlie,Chemistry,75
104,David,Math,88
104,David,Chemistry,88
104,David,Physics,95


In [0]:

# Filter only Math & Chemistry Subjects

from pyspark.sql.functions import *

filtered_records = df.where(col("Subject").isin("Math","Chemistry"))

#
filtered_records.display()


Student_ID,Name,Subject,Marks
101,Alice,Math,85
101,Alice,Chemistry,85
102,Bob,Math,90
103,Charlie,Math,75
103,Charlie,Chemistry,75
104,David,Math,88
104,David,Chemistry,88
105,Eve,Chemistry,91


In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *

# Rank the students based on their marks in Math and Chemistry

windowSpec = Window.partitionBy("Student_ID", "Name").orderBy(desc("Subject"))
df_lagged = filtered_records.withColumn("Previous_Marks", lag("Marks", 1).over(windowSpec))
df_lagged.display()


Student_ID,Name,Subject,Marks,Previous_Marks
101,Alice,Math,85,null
101,Alice,Chemistry,85,85
102,Bob,Math,90,null
103,Charlie,Math,75,null
103,Charlie,Chemistry,75,75
104,David,Math,88,null
104,David,Chemistry,88,88
105,Eve,Chemistry,91,null


In [0]:
df_lagged.filter(df_lagged.Previous_Marks == df_lagged.Marks).select("Student_ID", "Name").display()

Student_ID,Name
101,Alice
103,Charlie
104,David
